# Notebook: un test set

Giorno 3, mattina. 45 minuti, ognuno al proprio portatile.

Cinque modelli di visione su un piccolo test set: per ognuno un'esecuzione e una misura. I fotogrammi sono foto vere
di strade e parcheggi, con persone e targhe sfocate; i documenti del fascicolo e le targhe di fantasia sono sintetici.

| Parte | Minuti | Che cosa si misura |
|---|---|---|
| 1. Rilevamento | 9 | Precisione e recall a tre soglie |
| 2. Gruppi | 9 | La recall di giorno e di notte, con il numero di veicoli |
| 3. OCR | 9 | Il tasso di errore sui caratteri, l'IBAN esatto o no, i caratteri incerti |
| 4. Ricerca per immagini | 9 | Una frase cerca tra i fotogrammi: il veicolo è tra i primi cinque? |
| 5. VLM | 9 | Una targa troppo piccola, poi a piena risoluzione |
| 6. La tabella | | I vostri numeri, da incollare nel foglio condiviso |

Le celle si eseguono in ordine, con Maiusc+Invio. Dove trovate **Da fare**, cambiate la cella e rieseguitela. La
parte 2 usa i risultati della 1; le parti 3, 4 e 5 partono da sole: chi resta indietro salta alla parte successiva.

## 1. Rilevamento

Il test set: 29 fotogrammi da COCO, una raccolta pubblica di foto usata per misurare i modelli di visione. Per ogni
foto delle persone hanno disegnato i riquadri di auto, camion e autobus: è la verità con cui si confronta il modello.

Il modello di rilevamento è RT-DETR, addestrato su altre foto della stessa raccolta: queste non le ha mai viste. Per
ogni veicolo restituisce un riquadro e un punteggio da 0 a 1. La prima volta la cella carica il modello (qualche
secondo).

In [ ]:
from visione import SOGLIE, ModelloDiRilevamento, anteprima, apri, disegna, fotogrammi, misura, veicoli_veri

foto = fotogrammi()
veri = veicoli_veri()
print(len(foto), "fotogrammi,", sum(len(riquadri) for riquadri in veri.values()), "veicoli veri\n")

modello = ModelloDiRilevamento()
rilevamenti = {file: modello(file) for file in foto}
for riquadro, punteggio in rilevamenti["F07.jpg"][:5]:
    print(f"punteggio {punteggio:.2f}   riquadro {riquadro}")

Ogni riga è un veicolo trovato in `F07.jpg`: il punteggio, e il riquadro in pixel (sinistra, alto, destra, basso).

Un riquadro trovato sopra la soglia è giusto (TP) se si sovrappone a un veicolo vero per almeno metà: IoU
(sovrapposizione) di almeno 0,5. Altrimenti è un falso allarme (FP): un riquadro su niente, un doppione dello stesso
veicolo, o uno che lo copre per meno di metà. Un veicolo vero che nessun riquadro copre è un FN.

In [ ]:
disegna("F07.jpg", rilevamenti["F07.jpg"], veri["F07.jpg"], soglia=0.5)

Verde: TP. Rosso: FP. Giallo: FN, il veicolo che il modello non ha trovato.

**Da fare.** Guardate un altro fotogramma, da `F01.jpg` a `F29.jpg`, e cambiate la soglia. Che cosa cambia?

In [ ]:
disegna("F15.jpg", rilevamenti["F15.jpg"], veri["F15.jpg"], soglia=0.3)

Adesso tutto il test set, a tre soglie.

In [ ]:
def riga(nome, m):
    print(
        f"{nome:<8} {m['fotogrammi']:>10} {m['oggetti']:>7} {m['TP']:>4} {m['FP']:>4} {m['FN']:>4}"
        f" {m['precisione']:>10.2f} {m['recall']:>6.2f}"
    )


def intestazione(prima):
    print(
        f"{prima:<8} {'fotogrammi':>10} {'veicoli':>7} {'TP':>4} {'FP':>4} {'FN':>4} {'precisione':>10} {'recall':>6}"
    )


intestazione("soglia")
per_soglia = {soglia: misura(rilevamenti, veri, soglia) for soglia in SOGLIE}
for soglia, m in per_soglia.items():
    riga(str(soglia), m)

- Alzando la soglia, la precisione sale o scende? E la recall?
- Per il triage di foto con armi quale soglia scegliereste? E per un allarme che fa fermare un'auto?
- Con la soglia a 0,3 i FP sono tanti: aprite due fotogrammi con `disegna(...)`. Sono oggetti scambiati per veicoli,
  doppioni, o riquadri messi male?

## 2. Gruppi

La stessa misura divisa per gruppo, alla soglia di 0,5: 24 fotogrammi di giorno, 5 di notte o al crepuscolo. Accanto
alla recall, quanti veicoli ha ogni gruppo.

In [ ]:
SOGLIA = 0.5
per_gruppo = {}
intestazione("gruppo")
for gruppo in ("giorno", "notte"):
    file_del_gruppo = [file for file, g in foto.items() if g == gruppo]
    per_gruppo[gruppo] = misura(rilevamenti, veri, SOGLIA, file_del_gruppo)
    riga(gruppo, per_gruppo[gruppo])

veicoli_di_notte = per_gruppo["notte"]["oggetti"]
print(f"\nDi notte un veicolo trovato in più o in meno sposta la recall di {1 / veicoli_di_notte:.2f}.")

- La recall di notte è più bassa? Di quanto?
- Con quanti veicoli è misurata? Basta per dire che il modello vede peggio di notte?
- Guardate i FN di notte, in giallo qui sotto: sono veicoli al buio, o piccoli e lontani? I fotogrammi di notte
  cambiano anche in altro: pioggia, distanza, tipo di veicolo.

In [ ]:
for file, gruppo in foto.items():
    if gruppo == "notte":
        display(anteprima(disegna(file, rilevamenti[file], veri[file], SOGLIA), 480))

## 3. OCR

Tre documenti del fascicolo stampati come scansioni, un po' storti, sfocati e con un timbro: DOC-01, DOC-05
(l'estratto conto) e DOC-10. Poi FOTO-01: il primo IBAN di DOC-05 scritto a mano e fotografato.

L'OCR lavora in due passi: trova le zone di testo, poi le legge carattere per carattere. A ogni carattere dà una
confidenza.

In [ ]:
from visione import (
    DATI,
    FOTO_IBAN,
    Lettore,
    anteprima,
    apri,
    cer,
    differenze,
    iban_letto,
    iban_vero,
    incerti,
    riga_letta,
    testo,
)

lettore = Lettore()
display(anteprima(apri("DOC-05.jpg").crop((0, 0, 1240, 560)), 900))  # la parte scritta della pagina
letture = {"DOC-05": lettore(DATI / "scansioni" / "DOC-05.jpg")}
print(testo(letture["DOC-05"]))

Confrontate il testo letto con l'immagine. Il tasso di errore sui caratteri (CER) conta i caratteri sostituiti,
cancellati e inseriti, diviso i caratteri del testo vero. I caratteri incerti sono quelli con confidenza sotto 0,9.

In [ ]:
cer_per_documento = {}
print("documento   CER    caratteri incerti")
for documento in ("DOC-01", "DOC-05", "DOC-10"):
    letture[documento] = lettore(DATI / "scansioni" / f"{documento}.jpg")
    vero = (DATI / "scansioni" / f"{documento}.txt").read_text(encoding="utf-8")
    cer_per_documento[documento] = cer(testo(letture[documento]), vero)
    print(f"{documento}      {cer_per_documento[documento]:.1%}   {len(incerti(letture[documento]))}")

Un CER del 3% sembra poco. Ma un IBAN conta solo se ogni carattere è giusto: la corrispondenza esatta. DOC-05 ha due
IBAN: quelli dei bonifici del 3 e del 5 marzo.

In [ ]:
def confronta_iban(nome, letto, vero):
    trovato = iban_letto(letto, vero)
    iban_esatti[nome] = differenze(trovato, vero) == 0
    print(f"{nome}\n  vero:  {vero.replace(' ', '')}\n  letto: {trovato}")
    print(f"  caratteri da correggere: {differenze(trovato, vero)}   esatto: {'sì' if iban_esatti[nome] else 'no'}\n")


iban_esatti = {}
confronta_iban("DOC-05, bonifico del 3 marzo", riga_letta(letture["DOC-05"], "03/03/2025"), iban_vero(6))
confronta_iban("DOC-05, bonifico del 5 marzo", riga_letta(letture["DOC-05"], "05/03/2025"), iban_vero(8))
print("Le righe della tabella, con i caratteri incerti tra parentesi quadre:")
print("\n".join(testo(letture["DOC-05"], soglia=0.9).splitlines()[-5:]))

Adesso FOTO-01, scritto a mano: è lo stesso IBAN del bonifico del 3 marzo.

In [ ]:
from PIL import Image

display(anteprima(Image.open(FOTO_IBAN), 900))
letture["FOTO-01"] = lettore(FOTO_IBAN)
confronta_iban("FOTO-01", testo(letture["FOTO-01"]), iban_vero(6))
print("Con i caratteri incerti tra parentesi quadre:")
print(testo(letture["FOTO-01"], soglia=0.9))

- Quanti IBAN su tre sono esatti?
- I caratteri sbagliati sono tra quelli incerti? Una persona che controlla solo gli incerti li avrebbe trovati tutti?
- L'IBAN del 5 marzo ha due cifre scambiate fin dall'origine: è l'errore del fascicolo. L'OCR l'ha letto com'è
  scritto, o l'ha «corretto»?

## 4. Ricerca per immagini

Il modello (SigLIP 2) mette testo e immagini nello stesso spazio: ogni fotogramma diventa un vettore, e anche una
frase. I fotogrammi più vicini alla frase vengono per primi. È addestrato su coppie di immagini e didascalie in molte
lingue: la frase si scrive in italiano.

In [ ]:
from visione import Ricerca, fotogrammi, striscia

foto = fotogrammi()

ricerca = Ricerca()
ricerca.indicizza(list(foto))


def cerca(frase, k=5):
    trovati = ricerca.cerca(frase, k)
    print(frase, "->", ", ".join(f"{file} ({punteggio:.2f})" for file, punteggio in trovati))
    display(striscia([file for file, _ in trovati]))


cerca("un furgone dei gelati")

Il banco: cinque frasi, ciascuna con i suoi fotogrammi giusti. Il veicolo cercato è tra i primi cinque (recall@5)? E al
primo posto?

In [ ]:
BANCO = {
    "un furgone dei gelati": {"F23.jpg"},
    "un autobus rosso a due piani": {"F14.jpg", "F19.jpg"},
    "uno scuolabus giallo": {"F12.jpg"},
    "un taxi giallo": {"F01.jpg"},
    "un tram": {"F22.jpg"},
}
nei_primi_5 = al_primo_posto = 0
for frase, giusti in BANCO.items():
    primi = [file for file, _ in ricerca.cerca(frase, 5)]
    trovato = bool(giusti & set(primi))
    nei_primi_5 += trovato
    al_primo_posto += primi[0] in giusti
    print(f"{frase:<30} primo: {primi[0]}   nei primi 5: {'sì' if trovato else 'no'}")
print(f"\nrecall@5: {nei_primi_5} su {len(BANCO)}   al primo posto: {al_primo_posto} su {len(BANCO)}")

**Da fare.**

- Cercate qualcosa che nel test set non c'è, per esempio un'ambulanza. Che cosa torna?
- Riscrivete una frase del banco con altre parole, o in inglese. Il fotogramma giusto resta tra i primi cinque?

In [ ]:
cerca("un'ambulanza")

La ricerca restituisce sempre cinque risultati, anche quando nessuno va bene: l'unico segnale è un punteggio più
basso.

## 5. VLM

Un modello visivo-linguistico (VLM) piccolo: SmolVLM, 256 milioni di parametri. Riceve un'immagine e una domanda,
risponde in parole. Prima di entrare nel modello, l'immagine intera è rimpicciolita a 512 pixel sul lato lungo.

Le tre targhe sono di fantasia e non valide: le targhe italiane non usano le lettere I, O, Q e U. La domanda è in
inglese perché il modello è addestrato in inglese: in italiano tende a descrivere la foto invece di rispondere.

In [ ]:
from visione import ModelloVisivo, anteprima, apri, ritaglio, targhe

vlm = ModelloVisivo()
DOMANDA = "Read the license plate. Answer with the plate text only."


def uguali(letta, vera):
    return "".join(c for c in letta.upper() if c.isalnum()) == vera.replace(" ", "")


letture_vlm = []
for targa in targhe():
    intera, vicina = apri(targa["file"]), ritaglio(targa["file"], targa["riquadro"])
    display(anteprima(intera, 480), vicina)
    da_intera, da_ritaglio = vlm.chiedi(intera, DOMANDA), vlm.chiedi(vicina, DOMANDA)
    letture_vlm.append((uguali(da_intera, targa["targa"]), uguali(da_ritaglio, targa["targa"])))
    print(f"targa vera:       {targa['targa']}")
    print(f"foto intera:      {da_intera}   esatta: {'sì' if letture_vlm[-1][0] else 'no'}")
    print(f"ritaglio nitido:  {da_ritaglio}   esatta: {'sì' if letture_vlm[-1][1] else 'no'}\n")

- Sulla foto intera il modello ha detto che non si legge, o ha risposto comunque?
- Sul ritaglio le letture sono esatte? Il modello si accorge che le targhe non sono valide?

**Da fare.** Chiedete al modello, sulla stessa foto intera, se la targa si legge.

In [ ]:
vlm.chiedi(apri("T1.jpg"), "Can you read the license plate? Answer yes or no.")

Se risponde di no, perché prima ha dato una risposta? Il modello non sa di non vedere: a «leggi la targa» risponde
comunque con un testo, che va controllato come una lettura qualunque.

## 6. La tabella

Incollate queste righe nel foglio condiviso: le vediamo insieme sulla prossima slide. Una parte saltata resta vuota.

In [ ]:
righe = []
if "per_soglia" in globals():
    for soglia, m in per_soglia.items():
        righe += [
            ("rilevamento", f"precisione, soglia {soglia}", f"{m['precisione']:.2f}"),
            ("rilevamento", f"recall, soglia {soglia}", f"{m['recall']:.2f}"),
        ]
if "per_gruppo" in globals():
    for gruppo, m in per_gruppo.items():
        righe += [
            ("gruppi", f"recall {gruppo}, soglia {SOGLIA}", f"{m['recall']:.2f}"),
            ("gruppi", f"veicoli {gruppo}", str(m["oggetti"])),
        ]
if "cer_per_documento" in globals():
    righe += [("OCR", f"CER {documento}", f"{valore:.1%}") for documento, valore in cer_per_documento.items()]
if "iban_esatti" in globals():
    righe.append(("OCR", "IBAN esatti", f"{sum(iban_esatti.values())} su {len(iban_esatti)}"))
if "nei_primi_5" in globals():
    righe.append(("ricerca", "recall@5", f"{nei_primi_5} su {len(BANCO)}"))
if "letture_vlm" in globals():
    righe += [
        ("VLM", "targhe esatte, foto intera", f"{sum(a for a, _ in letture_vlm)} su {len(letture_vlm)}"),
        ("VLM", "targhe esatte, ritaglio", f"{sum(b for _, b in letture_vlm)} su {len(letture_vlm)}"),
    ]
print("parte\tmisura\tvalore")
print("\n".join("\t".join(r) for r in righe))

Tre domande per il confronto:

1. Quale soglia per quale uso, e perché?
2. Il divario di notte è un risultato o un indizio? Che cosa servirebbe per concluderlo?
3. Per un'indagine, quale errore preoccupa di più: un IBAN letto con una cifra sbagliata, o una targa inventata?